
## Phân chia tập dữ liệu Train / Val / Test theo Group (Chống Data Leakage)

### Mục tiêu:
1. **Căn cứ chọn dữ liệu**: Dựa trên kết luận từ `scripts/06_dataset_compare.ipynb`, **PKU-Market-PCB** là dataset chính duy nhất được chọn để huấn luyện, đánh giá và kiểm thử. DeepPCB không được gộp vào train/val/test.
2. **Cấu hình baseline**: Đọc `seed = 42` và `split_ratio = [0.7, 0.15, 0.15]` từ `configs/base.yaml`.
3. **Chia split theo group**: Sử dụng `sklearn.model_selection.GroupShuffleSplit` theo 2 bước:
   - Bước 1: Tách tập test 15% theo `group_id`.
   - Bước 2: Từ phần còn lại, tách tập val theo tỷ lệ `0.15 / (0.7 + 0.15) = 0.15 / 0.85` theo `group_id`.
   - Phần còn lại là tập train (70%).
4. **Kiểm tra rò rỉ group**: Khẳng định giao tập `group_id` giữa `train`, `val`, `test` là rỗng (`assert overlap == empty`).
5. **Xuất file split**: Lưu `data/splits/train.txt`, `val.txt`, `test.txt` với đường dẫn tương đối từ repo root, sắp xếp ổn định.
6. **Khóa split bằng Manifest**: Tạo `data/splits/split_manifest.json` ghi nhận seed, ngày tạo (ISO 8601), tỷ lệ, số ảnh, số group và mã MD5 checksum của 3 file split txt.

In [1]:
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
import yaml

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return current

ROOT = find_project_root()
print(f"Project root: {ROOT}")


Project root: /Users/Project/kltn-pcb


In [2]:
CONFIG_PATH = ROOT / "configs/base.yaml"
GROUPS_PATH = ROOT / "data/splits/groups.csv"
SPLITS_DIR = ROOT / "data/splits"

TRAIN_TXT = SPLITS_DIR / "train.txt"
VAL_TXT = SPLITS_DIR / "val.txt"
TEST_TXT = SPLITS_DIR / "test.txt"
MANIFEST_PATH = SPLITS_DIR / "split_manifest.json"

# Đọc cấu hình từ configs/base.yaml
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    base_config = yaml.safe_load(f)

SEED = int(base_config.get("seed", 42))
SPLIT_RATIO = base_config.get("split_ratio", [0.7, 0.15, 0.15])
train_ratio, val_ratio, test_ratio = [float(r) for r in SPLIT_RATIO]

print(f"Cấu hình từ {CONFIG_PATH.relative_to(ROOT)}:")
print(f"  Seed        : {SEED}")
print(f"  Split ratio : {SPLIT_RATIO} (Train: {train_ratio:.0%}, Val: {val_ratio:.0%}, Test: {test_ratio:.0%})")

# Đọc groups.csv và lọc dataset theo quyết định từ scripts/06_dataset_compare.ipynb
df_groups = pd.read_csv(GROUPS_PATH, dtype={"group_id": str})
total_raw_rows = len(df_groups)

# Theo kết luận scripts/06_dataset_compare.ipynb: PKU-Market-PCB là nguồn DUY NHẤT
df_selected = df_groups[df_groups["image_path"].str.contains("PKU-Market-PCB")].copy().reset_index(drop=True)
print(f"\nĐọc {total_raw_rows} bản ghi từ {GROUPS_PATH.relative_to(ROOT)}.")
print(f"Dataset được chọn: PKU-Market-PCB ({len(df_selected)} ảnh, {df_selected['group_id'].nunique()} groups).")
print(f"Danh sách groups: {sorted(df_selected['group_id'].unique())}")


Cấu hình từ configs/base.yaml:
  Seed        : 41
  Split ratio : [0.7, 0.15, 0.15] (Train: 70%, Val: 15%, Test: 15%)

Đọc 4837 bản ghi từ data/splits/groups.csv.
Dataset được chọn: PKU-Market-PCB (4837 ảnh, 10 groups).
Danh sách groups: ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12']


In [3]:
# Cảnh báo nếu các file split đích đã tồn tại
for target_file in [TRAIN_TXT, VAL_TXT, TEST_TXT, MANIFEST_PATH]:
    if target_file.exists():
        print(f"[CẢNH BÁO] File {target_file.relative_to(ROOT)} đã tồn tại. Sẽ được ghi đè (overwrite).")

# Bước 1: Tách test 15% theo group
gss_test = GroupShuffleSplit(n_splits=1, test_size=test_ratio, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(df_selected, groups=df_selected["group_id"].values))

df_train_val = df_selected.iloc[train_val_idx].copy().reset_index(drop=True)
df_test = df_selected.iloc[test_idx].copy().reset_index(drop=True)

# Bước 2: Từ phần còn lại (train_val), tách val theo tỷ lệ val_ratio / (train_ratio + val_ratio)
val_relative_ratio = val_ratio / (train_ratio + val_ratio)
gss_val = GroupShuffleSplit(n_splits=1, test_size=val_relative_ratio, random_state=SEED)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val["group_id"].values))

df_train = df_train_val.iloc[train_idx].copy().reset_index(drop=True)
df_val = df_train_val.iloc[val_idx].copy().reset_index(drop=True)

# Kiểm tra rò rỉ group giữa train/val/test
groups_train = set(df_train["group_id"].unique())
groups_val = set(df_val["group_id"].unique())
groups_test = set(df_test["group_id"].unique())

overlap_tr_val = groups_train & groups_val
overlap_tr_te = groups_train & groups_test
overlap_va_te = groups_val & groups_test

if overlap_tr_val or overlap_tr_te or overlap_va_te:
    raise RuntimeError(
        f"PHÁT HIỆN DATA LEAKAGE GIỮA CÁC SPLIT!\n"
        f"  Overlap Train - Val : {overlap_tr_val}\n"
        f"  Overlap Train - Test: {overlap_tr_te}\n"
        f"  Overlap Val - Test  : {overlap_va_te}"
    )

print("[PASS] Kiểm tra không rò rỉ group: Thành công (Giao giữa các tập = rỗng)")
print(f"  Train: {len(df_train):>5,} ảnh | {len(groups_train):>2} groups ({sorted(groups_train)})")
print(f"  Val  : {len(df_val):>5,} ảnh | {len(groups_val):>2} groups ({sorted(groups_val)})")
print(f"  Test : {len(df_test):>5,} ảnh | {len(groups_test):>2} groups ({sorted(groups_test)})")


[CẢNH BÁO] File data/splits/train.txt đã tồn tại. Sẽ được ghi đè (overwrite).
[CẢNH BÁO] File data/splits/val.txt đã tồn tại. Sẽ được ghi đè (overwrite).
[CẢNH BÁO] File data/splits/test.txt đã tồn tại. Sẽ được ghi đè (overwrite).
[CẢNH BÁO] File data/splits/split_manifest.json đã tồn tại. Sẽ được ghi đè (overwrite).
[PASS] Kiểm tra không rò rỉ group: Thành công (Giao giữa các tập = rỗng)
  Train: 3,989 ảnh |  6 groups (['01', '04', '05', '06', '07', '12'])
  Val  :   368 ảnh |  2 groups (['10', '11'])
  Test :   480 ảnh |  2 groups (['08', '09'])


In [4]:
# Lấy danh sách đường dẫn ảnh tương đối và sắp xếp ổn định
train_paths = sorted(df_train["image_path"].tolist())
val_paths = sorted(df_val["image_path"].tolist())
test_paths = sorted(df_test["image_path"].tolist())

def write_split_file(filepath: Path, paths: list[str]) -> None:
    with open(filepath, "w", encoding="utf-8") as f:
        for p in paths:
            f.write(f"{p}\n")

write_split_file(TRAIN_TXT, train_paths)
write_split_file(VAL_TXT, val_paths)
write_split_file(TEST_TXT, test_paths)

print(f"Đã lưu danh sách đường dẫn ảnh vào:")
print(f"  - {TRAIN_TXT.relative_to(ROOT)} ({len(train_paths):,} ảnh)")
print(f"  - {VAL_TXT.relative_to(ROOT)}   ({len(val_paths):,} ảnh)")
print(f"  - {TEST_TXT.relative_to(ROOT)}  ({len(test_paths):,} ảnh)")


Đã lưu danh sách đường dẫn ảnh vào:
  - data/splits/train.txt (3,989 ảnh)
  - data/splits/val.txt   (368 ảnh)
  - data/splits/test.txt  (480 ảnh)


In [5]:
def compute_md5(filepath: Path) -> str:
    hasher = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

md5_checksums = {
    "train.txt": compute_md5(TRAIN_TXT),
    "val.txt": compute_md5(VAL_TXT),
    "test.txt": compute_md5(TEST_TXT),
}

manifest_data = {
    "dataset": "PKU-Market-PCB",
    "dataset_selection_rationale": "Selected PKU-Market-PCB as the single source based on compatibility analysis in scripts/06_dataset_compare.ipynb (0/5 criteria met for merging DeepPCB).",
    "group_source_file": str(GROUPS_PATH.relative_to(ROOT)),
    "seed": SEED,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "split_ratio_target": SPLIT_RATIO,
    "split_strategy": "GroupShuffleSplit",
    "num_images": {
        "train": len(train_paths),
        "val": len(val_paths),
        "test": len(test_paths),
        "total": len(train_paths) + len(val_paths) + len(test_paths),
    },
    "num_groups": {
        "train": len(groups_train),
        "val": len(groups_val),
        "test": len(groups_test),
        "total": len(groups_train) + len(groups_val) + len(groups_test),
    },
    "groups": {
        "train": sorted(groups_train),
        "val": sorted(groups_val),
        "test": sorted(groups_test),
    },
    "md5_checksums": md5_checksums,
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest_data, f, indent=2, ensure_ascii=False)

print(f"Đã tạo {MANIFEST_PATH.relative_to(ROOT)}:\n")
print(json.dumps(manifest_data, indent=2, ensure_ascii=False))


Đã tạo data/splits/split_manifest.json:

{
  "dataset": "PKU-Market-PCB",
  "dataset_selection_rationale": "Selected PKU-Market-PCB as the single source based on compatibility analysis in scripts/06_dataset_compare.ipynb (0/5 criteria met for merging DeepPCB).",
  "group_source_file": "data/splits/groups.csv",
  "seed": 41,
  "created_at": "2026-09-19T11:46:24.388436+00:00",
  "split_ratio_target": [
    0.7,
    0.15,
    0.15
  ],
  "split_strategy": "GroupShuffleSplit",
  "num_images": {
    "train": 3989,
    "val": 368,
    "test": 480,
    "total": 4837
  },
  "num_groups": {
    "train": 6,
    "val": 2,
    "test": 2,
    "total": 10
  },
  "groups": {
    "train": [
      "01",
      "04",
      "05",
      "06",
      "07",
      "12"
    ],
    "val": [
      "10",
      "11"
    ],
    "test": [
      "08",
      "09"
    ]
  },
  "md5_checksums": {
    "train.txt": "12fffda36e6850eb2912fa0d0a18c494",
    "val.txt": "9246d70f17bf0527d4676d9fee31c347",
    "test.txt": "9ea11

In [6]:
# Kiểm tra tính toàn vẹn (Integrity Check)
print("=" * 60)
print("KIỂM TRA TÍNH TOÀN VẸN VÀ TỒN TẠI FILE")
print("=" * 60)

all_split_paths = train_paths + val_paths + test_paths
missing_images = [p for p in all_split_paths if not (ROOT / p).is_file()]
print(f"Tổng số ảnh trong 3 split: {len(all_split_paths):,}")
print(f"Ảnh không tồn tại trên đĩa : {len(missing_images)}")
assert len(missing_images) == 0, f"Có {len(missing_images)} ảnh bị thiếu!"

# Kiểm tra nhãn tương ứng đã tạo ở bước 07
missing_labels = 0
for img_p in all_split_paths:
    rel = Path(img_p).relative_to("data/raw/PKU-Market-PCB(Data enhanced version)")
    parts = list(rel.parts)
    if "images" in parts:
        parts.remove("images")
    lbl_p = ROOT / "data/processed/labels" / Path(*parts).with_suffix(".txt")
    if not lbl_p.is_file():
        missing_labels += 1

print(f"Nhãn trong data/processed/labels bị thiếu: {missing_labels}")
assert missing_labels == 0, f"Có {missing_labels} file nhãn chưa được tạo!"
print("-" * 60)
print("[PASS] Tất cả ảnh và file nhãn tương ứng đều tồn tại 100%!")
print("=" * 60)


KIỂM TRA TÍNH TOÀN VẸN VÀ TỒN TẠI FILE
Tổng số ảnh trong 3 split: 4,837
Ảnh không tồn tại trên đĩa : 0


Nhãn trong data/processed/labels bị thiếu: 0
------------------------------------------------------------
[PASS] Tất cả ảnh và file nhãn tương ứng đều tồn tại 100%!
